In [1]:
!pip install -q scipy scikit-learn pandas numpy

In [2]:
import shutil
shutil.copy(
    "/kaggle/input/datasets/just7nik/r2-datasets-p3-analysis/baseline_results (2).csv",
    "/kaggle/working/baseline_results.csv"
)

'/kaggle/working/baseline_results.csv'

In [3]:
%%writefile phase3_analysis_v2.py
"""
phase3_analysis.py  v2
======================
Fragility Under Constraint — Phase 3 & 4: Statistical Analysis + DPS

CHANGES IN V2
-------------
1. Bootstrap 95% confidence intervals added to all ΔAcc results
   (n=1000 bootstrap samples per cell, reported as [ci_low, ci_high])
2. Friedman test updated to include "mid" severity (noise has 3 levels now)
3. SmolLM Financial NLP excluded from DPS fitting (baseline=0.0 degenerate)
4. Latency-accuracy Pareto analysis added as new Section 8
5. DPS dataclass extended with ci_low/ci_high fields

What this script does
---------------------
1. Statistical tests
   - Wilcoxon signed-rank test: per model, is ΔAcc significantly > 0?
   - Friedman test: across all models, is shift sensitivity ranking consistent?
   - Spearman correlation: between baseline accuracy and shift robustness
   - Bootstrap 95% CI on mean ΔAcc per model × shift × domain

2. Degradation profiling (Phase 3)
   - Degradation slope per model × shift type (cliff vs gradual)
   - Cross-tier gap: how much does quantization amplify shift sensitivity?
   - Now uses low/mid/high for noise, low/high for prompt_degradation+vocab

3. DPS feature construction (Phase 4 input)
   - BaselinePerf, ShiftSensitivity, QuantizationGap, CalibrationScore
   - Excludes degenerate cells (baseline_acc < 0.1) from fitting
   - OLS regression + Spearman rank correlation on held-out scenarios

4. Latency-accuracy Pareto analysis (new)
   - Identifies deployment sweet spot per domain

5. Output files
   - phase3_stats.json       — all statistical test results + CIs
   - phase3_profiles.csv     — degradation profiles
   - phase4_dps.csv          — DPS scores and rankings
   - phase4_regression.json  — OLS weights and validation metrics
   - phase3_pareto.csv       — latency-accuracy Pareto data
   - phase3_report.txt       — human-readable summary

Usage
-----
python phase3_analysis.py \
    --baseline  /kaggle/working/baseline_results.csv \
    --shifts    /kaggle/working/shift_results_final.csv \
    --output-dir /kaggle/working
"""

import argparse
import json
import os
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
def import_deps():
    try:
        import numpy as np
        import pandas as pd
        from scipy import stats
        from sklearn.linear_model import LinearRegression
        from sklearn.preprocessing import StandardScaler
        return np, pd, stats, LinearRegression, StandardScaler
    except ImportError as e:
        print(f"Missing dependency: {e}")
        print("Run: pip install numpy pandas scipy scikit-learn")
        sys.exit(1)


# ---------------------------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------------------------
@dataclass
class DegradationProfile:
    model_key: str
    model_name: str
    shift_type: str
    domain: str
    slope: float           # |Δacc_high - Δacc_low| (cliff = high slope)
    cliff_type: str        # "cliff" if slope > threshold, "gradual" otherwise
    mean_delta_low: float
    mean_delta_high: float
    ci_low_low: float      # 95% CI lower bound on low-severity ΔAcc
    ci_high_low: float     # 95% CI upper bound on low-severity ΔAcc
    ci_low_high: float     # 95% CI lower bound on high-severity ΔAcc
    ci_high_high: float    # 95% CI upper bound on high-severity ΔAcc
    cross_tier_gap: float  # max(ΔAcc across tiers) - min(ΔAcc across tiers)
    tier_amplification: bool  # True if INT4 degrades more than FP16


@dataclass
class DPSEntry:
    model_key: str
    model_name: str
    tier: str
    domain: str
    baseline_perf: float
    shift_sensitivity: float   # mean |ΔAcc| across all shifts
    quantization_gap: float    # FP16_acc - tier_acc at baseline
    calibration_score: float   # 1 - ECE_proxy (higher = better calibrated)
    dps_raw: float             # weighted composite score
    dps_rank: int              # rank within scenario (1=best)


# ---------------------------------------------------------------------------
# Bootstrap CI helper
# ---------------------------------------------------------------------------
def bootstrap_ci(values, n_boot: int = 1000, ci: float = 0.95, seed: int = 42):
    """
    Compute bootstrap confidence interval for the mean of values.
    Returns (ci_lower, ci_upper).
    """
    import numpy as np
    rng = np.random.default_rng(seed)
    vals = np.array(values)
    if len(vals) == 0:
        return (0.0, 0.0)
    boot_means = np.array([
        rng.choice(vals, size=len(vals), replace=True).mean()
        for _ in range(n_boot)
    ])
    alpha = (1 - ci) / 2
    return (
        round(float(np.percentile(boot_means, alpha * 100)), 4),
        round(float(np.percentile(boot_means, (1 - alpha) * 100)), 4),
    )


# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
def load_data(baseline_path: str, shift_path: str, pd):
    baseline = pd.read_csv(baseline_path)
    shifts   = pd.read_csv(shift_path)

    # Normalize column types
    baseline["accuracy"]  = baseline["accuracy"].astype(float)
    baseline["f1_macro"]  = baseline["f1_macro"].astype(float)
    baseline["ece_proxy"] = baseline["ece_proxy"].astype(float)
    shifts["delta_acc"]   = shifts["delta_acc"].astype(float)
    shifts["delta_f1"]    = shifts["delta_f1"].astype(float)
    shifts["shifted_acc"] = shifts["shifted_acc"].astype(float)

    print(f"Baseline: {len(baseline)} rows | Shifts: {len(shifts)} rows")
    return baseline, shifts


# ---------------------------------------------------------------------------
# 1. Statistical tests
# ---------------------------------------------------------------------------
def run_statistical_tests(baseline, shifts, stats, np, pd) -> Dict:
    results = {}

    models     = sorted(shifts["model_key"].unique())
    shift_types= sorted(shifts["shift_type"].unique())
    domains    = sorted(shifts["domain"].unique())

    # -----------------------------------------------------------------------
    # 1a. Wilcoxon signed-rank: per model, is |Δacc| significantly > 0?
    # Tests whether each model degrades significantly under each shift type.
    # -----------------------------------------------------------------------
    wilcoxon_results = []
    for model in models:
        for st in shift_types:
            mask = (shifts["model_key"] == model) & (shifts["shift_type"] == st)
            deltas = shifts.loc[mask, "delta_acc"].values
            if len(deltas) < 5:
                continue
            ci_lo, ci_hi = bootstrap_ci(deltas)
            try:
                stat, p = stats.wilcoxon(deltas, alternative="greater", zero_method="wilcox")
                wilcoxon_results.append({
                    "model": model,
                    "shift_type": st,
                    "n": len(deltas),
                    "statistic": round(float(stat), 4),
                    "p_value": round(float(p), 4),
                    "significant": bool(p < 0.05),
                    "mean_delta": round(float(np.mean(deltas)), 4),
                    "ci_95_low":  ci_lo,
                    "ci_95_high": ci_hi,
                })
            except Exception as e:
                wilcoxon_results.append({
                    "model": model, "shift_type": st, "n": len(deltas),
                    "statistic": None, "p_value": None,
                    "significant": None,
                    "mean_delta": round(float(np.mean(deltas)), 4),
                    "ci_95_low": ci_lo, "ci_95_high": ci_hi,
                    "error": str(e)
                })
    results["wilcoxon"] = wilcoxon_results

    # -----------------------------------------------------------------------
    # 1b. Friedman test: across models, is shift sensitivity ranking consistent?
    # One Friedman test per shift_type × domain × severity combination.
    # -----------------------------------------------------------------------
    friedman_results = []
    for st in shift_types:
        for dom in domains:
            # v2 fix: use all severities present in data (noise has low/mid/high)
            all_sevs = sorted(shifts.loc[shifts["shift_type"] == st, "severity"].unique())
            for sev in all_sevs:
                groups = []
                group_models = []
                for model in models:
                    mask = (
                        (shifts["model_key"] == model) &
                        (shifts["shift_type"] == st) &
                        (shifts["domain"] == dom) &
                        (shifts["severity"] == sev)
                    )
                    vals = shifts.loc[mask, "delta_acc"].values
                    if len(vals) > 0:
                        groups.append(vals)
                        group_models.append(model)

                if len(groups) < 3 or any(len(g) < 2 for g in groups):
                    continue
                # Pad to equal length
                min_len = min(len(g) for g in groups)
                groups = [g[:min_len] for g in groups]
                try:
                    stat, p = stats.friedmanchisquare(*groups)
                    friedman_results.append({
                        "shift_type": st, "domain": dom, "severity": sev,
                        "models": group_models,
                        "statistic": round(float(stat), 4),
                        "p_value": round(float(p), 4),
                        "significant": bool(p < 0.05),
                    })
                except Exception as e:
                    friedman_results.append({
                        "shift_type": st, "domain": dom, "severity": sev,
                        "error": str(e)
                    })
    results["friedman"] = friedman_results

    # -----------------------------------------------------------------------
    # 1c. Spearman correlation: baseline accuracy vs mean |Δacc|
    # Does a better baseline predict better shift robustness?
    # -----------------------------------------------------------------------
    spearman_results = []
    for dom in domains:
        b_mask = baseline["domain"] == dom
        b_sub  = baseline.loc[b_mask, ["model_key", "tier", "accuracy"]].copy()

        for st in shift_types:
            s_mask = (shifts["domain"] == dom) & (shifts["shift_type"] == st)
            s_sub  = shifts.loc[s_mask, ["model_key", "tier", "delta_acc"]].copy()
            s_agg  = s_sub.groupby(["model_key", "tier"])["delta_acc"].mean().reset_index()
            s_agg.columns = ["model_key", "tier", "mean_delta"]

            merged = b_sub.merge(s_agg, on=["model_key", "tier"])
            if len(merged) < 5:
                continue

            try:
                rho, p = stats.spearmanr(merged["accuracy"], -merged["mean_delta"])
                spearman_results.append({
                    "domain": dom, "shift_type": st,
                    "n_pairs": len(merged),
                    "spearman_rho": round(float(rho), 4),
                    "p_value": round(float(p), 4),
                    "significant": bool(p < 0.05),
                    "interpretation": (
                        "Higher baseline → more robust" if rho > 0
                        else "Higher baseline → more fragile"
                    ),
                })
            except Exception as e:
                spearman_results.append({
                    "domain": dom, "shift_type": st, "error": str(e)
                })
    results["spearman"] = spearman_results

    return results


# ---------------------------------------------------------------------------
# 2. Degradation profiling
# ---------------------------------------------------------------------------
def compute_degradation_profiles(shifts, np, pd) -> List[DegradationProfile]:
    profiles = []
    CLIFF_THRESHOLD = 0.15  # slope > 0.15 = cliff degradation

    models      = sorted(shifts["model_key"].unique())
    shift_types = sorted(shifts["shift_type"].unique())
    domains     = sorted(shifts["domain"].unique())
    tiers       = ["fp16", "int8", "int4"]

    for model in models:
        model_name = shifts.loc[shifts["model_key"] == model, "model_name"].iloc[0]
        for st in shift_types:
            for dom in domains:
                # v2: use "low" and "high" for slope computation across all shift types.
                # For noise (which has low/mid/high), we use low vs high to get the
                # full severity range — mid is captured in the cross-tier analysis.
                low_mask  = ((shifts["model_key"] == model) & (shifts["shift_type"] == st) &
                             (shifts["domain"] == dom) & (shifts["severity"] == "low"))
                high_mask = ((shifts["model_key"] == model) & (shifts["shift_type"] == st) &
                             (shifts["domain"] == dom) & (shifts["severity"] == "high"))

                low_vals  = shifts.loc[low_mask,  "delta_acc"].values
                high_vals = shifts.loc[high_mask, "delta_acc"].values

                if len(low_vals) == 0 or len(high_vals) == 0:
                    continue

                mean_low  = float(np.mean(low_vals))
                mean_high = float(np.mean(high_vals))

                # Bootstrap CIs for low and high severity
                ci_lo_low,  ci_hi_low  = bootstrap_ci(low_vals)
                ci_lo_high, ci_hi_high = bootstrap_ci(high_vals)

                # Slope: how much does degradation worsen from low → high severity?
                slope = abs(mean_high - mean_low)
                cliff_type = "cliff" if slope >= CLIFF_THRESHOLD else "gradual"

                # Cross-tier gap: does quantization amplify shift sensitivity?
                tier_deltas = {}
                for tier in tiers:
                    t_mask = ((shifts["model_key"] == model) & (shifts["shift_type"] == st) &
                              (shifts["domain"] == dom) & (shifts["tier"] == tier))
                    t_vals = shifts.loc[t_mask, "delta_acc"].values
                    if len(t_vals) > 0:
                        tier_deltas[tier] = float(np.mean(t_vals))

                if len(tier_deltas) >= 2:
                    cross_tier_gap = round(max(tier_deltas.values()) - min(tier_deltas.values()), 4)
                    fp16_deg = tier_deltas.get("fp16", 0)
                    int4_deg = tier_deltas.get("int4", 0)
                    tier_amp = int4_deg > fp16_deg
                else:
                    cross_tier_gap = 0.0
                    tier_amp = False

                profiles.append(DegradationProfile(
                    model_key=model,
                    model_name=model_name,
                    shift_type=st,
                    domain=dom,
                    slope=round(slope, 4),
                    cliff_type=cliff_type,
                    mean_delta_low=round(mean_low, 4),
                    mean_delta_high=round(mean_high, 4),
                    ci_low_low=ci_lo_low,
                    ci_high_low=ci_hi_low,
                    ci_low_high=ci_lo_high,
                    ci_high_high=ci_hi_high,
                    cross_tier_gap=cross_tier_gap,
                    tier_amplification=tier_amp,
                ))

    return profiles


# ---------------------------------------------------------------------------
# 3. DPS construction
# ---------------------------------------------------------------------------
def compute_dps(baseline, shifts, profiles_df, LinearRegression, StandardScaler, np, pd,
                stats) -> Tuple[List[DPSEntry], Dict]:
    """
    DPS(model, tier, domain) = w1 × BaselinePerf
                              - w2 × ShiftSensitivity
                              - w3 × QuantizationGap
                              + w4 × CalibrationScore

    Weights learned via OLS regression predicting mean Δacc.
    Validated by Spearman ρ on held-out scenarios.
    """
    dps_entries = []
    regression_info = {}

    models  = sorted(baseline["model_key"].unique())
    tiers   = sorted(baseline["tier"].unique())
    domains = sorted(baseline["domain"].unique())

    # Build FP16 baseline per model × domain for QuantizationGap
    fp16_base = (baseline[baseline["tier"] == "fp16"]
                 .set_index(["model_key", "domain"])["accuracy"]
                 .to_dict())

    # v2 fix: exclude degenerate cells where baseline_acc < 0.1
    # SmolLM Financial NLP (baseline=0.0) distorts DPS weights
    DPS_MIN_BASELINE = 0.1

    rows = []
    for model in models:
        for tier in tiers:
            for domain in domains:
                b_mask = ((baseline["model_key"] == model) &
                          (baseline["tier"] == tier) &
                          (baseline["domain"] == domain))
                b_row = baseline[b_mask]
                if len(b_row) == 0:
                    continue

                baseline_perf = float(b_row["accuracy"].iloc[0])

                # Skip degenerate cases
                if baseline_perf < DPS_MIN_BASELINE:
                    continue

                ece = float(b_row["ece_proxy"].iloc[0])
                calibration   = 1.0 - min(ece, 1.0)

                # Quantization gap vs FP16
                fp16_acc = fp16_base.get((model, domain), baseline_perf)
                quant_gap = max(0.0, fp16_acc - baseline_perf)

                # Shift sensitivity: mean |Δacc| across all shifts for this cell
                s_mask = ((shifts["model_key"] == model) &
                          (shifts["tier"] == tier) &
                          (shifts["domain"] == domain))
                s_vals = shifts[s_mask]["delta_acc"].values
                if len(s_vals) == 0:
                    continue
                shift_sensitivity = float(np.mean(np.abs(s_vals)))
                mean_delta        = float(np.mean(s_vals))

                rows.append({
                    "model_key":         model,
                    "model_name":        b_row["model_name"].iloc[0],
                    "tier":              tier,
                    "domain":            domain,
                    "baseline_perf":     baseline_perf,
                    "shift_sensitivity": shift_sensitivity,
                    "quantization_gap":  quant_gap,
                    "calibration_score": calibration,
                    "mean_delta":        mean_delta,
                })

    if not rows:
        return [], {}

    df = pd.DataFrame(rows)

    # -----------------------------------------------------------------------
    # Train/validation split: 70/30 by scenario
    # -----------------------------------------------------------------------
    np.random.seed(42)
    n = len(df)
    idx = np.random.permutation(n)
    split = int(0.7 * n)
    train_idx = idx[:split]
    val_idx   = idx[split:]

    train = df.iloc[train_idx]
    val   = df.iloc[val_idx]

    feature_cols = ["baseline_perf", "shift_sensitivity", "quantization_gap", "calibration_score"]
    X_train = train[feature_cols].values
    y_train = train["mean_delta"].values
    X_val   = val[feature_cols].values
    y_val   = val["mean_delta"].values

    scaler  = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    reg = LinearRegression()
    reg.fit(X_train_s, y_train)

    # Weights (signed)
    weights = {col: round(float(w), 4) for col, w in zip(feature_cols, reg.coef_)}
    weights["intercept"] = round(float(reg.intercept_), 4)

    # Spearman rank correlation on validation set
    y_pred_val = reg.predict(X_val_s)
    rho_val, p_val = stats.spearmanr(-y_val, -y_pred_val)  # higher=better

    regression_info = {
        "n_train":        int(len(train)),
        "n_val":          int(len(val)),
        "weights":        weights,
        "r2_train":       round(float(reg.score(X_train_s, y_train)), 4),
        "r2_val":         round(float(reg.score(X_val_s, y_val)), 4),
        "spearman_rho":   round(float(rho_val), 4),
        "spearman_p":     round(float(p_val), 4),
        "interpretation": (
            f"Spearman ρ={rho_val:.3f} on held-out scenarios. "
            + ("Strong predictive validity." if abs(rho_val) >= 0.75
               else "Moderate predictive validity." if abs(rho_val) >= 0.5
               else "Weak predictive validity — DPS needs refinement.")
        ),
    }

    # -----------------------------------------------------------------------
    # Compute DPS scores for all rows
    # -----------------------------------------------------------------------
    X_all = scaler.transform(df[feature_cols].values)
    dps_scores = reg.predict(X_all)   # predicted Δacc (lower = more robust)

    for i, (_, row) in enumerate(df.iterrows()):
        dps_entries.append(DPSEntry(
            model_key=row["model_key"],
            model_name=row["model_name"],
            tier=row["tier"],
            domain=row["domain"],
            baseline_perf=round(row["baseline_perf"], 4),
            shift_sensitivity=round(row["shift_sensitivity"], 4),
            quantization_gap=round(row["quantization_gap"], 4),
            calibration_score=round(row["calibration_score"], 4),
            dps_raw=round(float(dps_scores[i]), 4),
            dps_rank=0,  # filled below
        ))

    # Rank within each domain (higher DPS raw = more predicted degradation = worse)
    # Rank 1 = most robust (lowest predicted degradation)
    dps_df = pd.DataFrame([asdict(e) for e in dps_entries])
    for dom in dps_df["domain"].unique():
        mask = dps_df["domain"] == dom
        ranks = dps_df.loc[mask, "dps_raw"].rank(ascending=True).astype(int)
        for idx_r, rank in ranks.items():
            dps_entries[idx_r].dps_rank = int(rank)

    return dps_entries, regression_info


# ---------------------------------------------------------------------------
# 4. Latency-Accuracy Pareto Analysis
# ---------------------------------------------------------------------------
def compute_pareto(baseline, np, pd):
    """
    For each model × tier combination, compute the accuracy-latency tradeoff
    across all domains. Identifies Pareto-optimal points (no other point
    has both higher accuracy AND lower latency).
    """
    agg = (baseline
           .groupby(["model_key", "model_name", "tier"])
           .agg(
               mean_accuracy=("accuracy", "mean"),
               mean_latency=("latency_ms_mean", "mean"),
               mean_f1=("f1_macro", "mean"),
           )
           .reset_index())

    # Mark Pareto-optimal points
    pareto_flags = []
    for i, row in agg.iterrows():
        dominated = False
        for j, other in agg.iterrows():
            if i == j:
                continue
            # other dominates row if other has >= accuracy AND <= latency
            if (other["mean_accuracy"] >= row["mean_accuracy"] and
                    other["mean_latency"] <= row["mean_latency"] and
                    (other["mean_accuracy"] > row["mean_accuracy"] or
                     other["mean_latency"] < row["mean_latency"])):
                dominated = True
                break
        pareto_flags.append(not dominated)

    agg["pareto_optimal"] = pareto_flags
    agg["mean_accuracy"]  = agg["mean_accuracy"].round(4)
    agg["mean_latency"]   = agg["mean_latency"].round(2)
    agg["mean_f1"]        = agg["mean_f1"].round(4)
    return agg.sort_values("mean_accuracy", ascending=False)


# ---------------------------------------------------------------------------
# 5. Human-readable report
# ---------------------------------------------------------------------------
def write_report(
    stats_results: Dict,
    profiles: List[DegradationProfile],
    dps_entries: List[DPSEntry],
    regression_info: Dict,
    pareto_df,
    np, pd,
) -> str:
    lines = []
    lines.append("=" * 70)
    lines.append("FRAGILITY UNDER CONSTRAINT — PHASE 3/4 ANALYSIS REPORT")
    lines.append("=" * 70)

    # --- Wilcoxon summary ---
    lines.append("\n[1] WILCOXON SIGNED-RANK TEST RESULTS")
    lines.append("    H0: mean Δacc = 0 (no degradation)")
    lines.append("    H1: mean Δacc > 0 (significant degradation)")
    lines.append("-" * 70)
    w_results = stats_results.get("wilcoxon", [])
    sig   = [r for r in w_results if r.get("significant")]
    insig = [r for r in w_results if r.get("significant") == False]
    lines.append(f"    Significant (p<0.05): {len(sig)}/{len(w_results)} tests")
    lines.append("")
    lines.append(f"    {'Model':15s} {'Shift':28s} {'Mean Δacc':9s} {'95% CI':18s} {'p-value':8s} {'Sig?'}")
    lines.append(f"    {'-'*15} {'-'*28} {'-'*9} {'-'*18} {'-'*8} {'-'*4}")
    for r in sorted(w_results, key=lambda x: x.get("mean_delta", 0), reverse=True):
        sig_str = "YES" if r.get("significant") else "no"
        p_str   = f"{r['p_value']:.4f}" if r.get("p_value") is not None else "N/A"
        ci_str  = f"[{r.get('ci_95_low',0):+.4f}, {r.get('ci_95_high',0):+.4f}]"
        lines.append(
            f"    {r['model']:15s} {r['shift_type']:28s} "
            f"{r.get('mean_delta', 0):+9.4f} {ci_str:18s} {p_str:8s} {sig_str}"
        )

    # --- Friedman summary ---
    lines.append("\n[2] FRIEDMAN TEST RESULTS")
    lines.append("    H0: model rankings under shift are consistent across domains")
    lines.append("-" * 70)
    f_results = stats_results.get("friedman", [])
    f_sig = [r for r in f_results if r.get("significant")]
    lines.append(f"    Significant (p<0.05): {len(f_sig)}/{len(f_results)} tests")
    for r in f_sig[:5]:
        lines.append(
            f"    {r['shift_type']:28s} | {r['domain']:22s} | {r['severity']:4s} | "
            f"χ²={r['statistic']:.2f} p={r['p_value']:.4f}"
        )

    # --- Spearman ---
    lines.append("\n[3] SPEARMAN CORRELATION: BASELINE ACCURACY → SHIFT ROBUSTNESS")
    lines.append("-" * 70)
    s_results = stats_results.get("spearman", [])
    for r in s_results:
        sig_str = "p<0.05" if r.get("significant") else "n.s."
        rho_str = f"{r['spearman_rho']:.3f}" if "spearman_rho" in r else "N/A"
        lines.append(
            f"    {r['domain']:22s} | {r['shift_type']:28s} | "
            f"ρ={rho_str} ({sig_str}) | {r.get('interpretation','')}"
        )

    # --- Degradation profiles ---
    lines.append("\n[4] DEGRADATION PROFILES — CLIFF vs GRADUAL")
    lines.append("-" * 70)
    profiles_df = pd.DataFrame([asdict(p) for p in profiles])
    cliff = profiles_df[profiles_df["cliff_type"] == "cliff"]
    lines.append(f"    Cliff degradation: {len(cliff)}/{len(profiles_df)} profiles")
    lines.append("")
    lines.append(f"    {'Model':15s} {'Shift':28s} {'Domain':22s} {'Type':8s} {'Slope':6s} {'Tier amp?'}")
    lines.append(f"    {'-'*15} {'-'*28} {'-'*22} {'-'*8} {'-'*6} {'-'*9}")
    for _, row in profiles_df.sort_values("slope", ascending=False).head(15).iterrows():
        lines.append(
            f"    {row['model_key']:15s} {row['shift_type']:28s} "
            f"{row['domain']:22s} {row['cliff_type']:8s} "
            f"{row['slope']:.3f}  {'YES' if row['tier_amplification'] else 'no'}"
        )

    # --- Cross-tier gap ---
    lines.append("\n[5] CROSS-TIER GAP (quantization amplifies shift sensitivity?)")
    lines.append("-" * 70)
    tier_amp = profiles_df[profiles_df["tier_amplification"] == True]
    lines.append(f"    Tier amplification detected: {len(tier_amp)}/{len(profiles_df)} profiles")
    if len(tier_amp) > 0:
        lines.append(f"    Mean cross-tier gap where amplified: "
                     f"{tier_amp['cross_tier_gap'].mean():.4f}")
        lines.append(f"    Max cross-tier gap: "
                     f"{profiles_df['cross_tier_gap'].max():.4f} "
                     f"({profiles_df.loc[profiles_df['cross_tier_gap'].idxmax(), 'model_key']} / "
                     f"{profiles_df.loc[profiles_df['cross_tier_gap'].idxmax(), 'shift_type']} / "
                     f"{profiles_df.loc[profiles_df['cross_tier_gap'].idxmax(), 'domain']})")

    # --- DPS ---
    lines.append("\n[6] DEGRADATION PREDICTION SCORE (DPS)")
    lines.append("-" * 70)
    lines.append(f"    Formula: DPS = w1×BaselinePerf - w2×ShiftSensitivity")
    lines.append(f"                  - w3×QuantizationGap + w4×CalibrationScore")
    lines.append("")
    if regression_info:
        w = regression_info.get("weights", {})
        lines.append(f"    Learned weights (OLS, standardized features):")
        lines.append(f"      w1 (BaselinePerf):     {w.get('baseline_perf', 'N/A'):+.4f}")
        lines.append(f"      w2 (ShiftSensitivity): {w.get('shift_sensitivity', 'N/A'):+.4f}")
        lines.append(f"      w3 (QuantizationGap):  {w.get('quantization_gap', 'N/A'):+.4f}")
        lines.append(f"      w4 (CalibrationScore): {w.get('calibration_score', 'N/A'):+.4f}")
        lines.append(f"      intercept:             {w.get('intercept', 'N/A'):+.4f}")
        lines.append("")
        lines.append(f"    R² train: {regression_info.get('r2_train', 'N/A'):.4f}")
        lines.append(f"    R² val:   {regression_info.get('r2_val', 'N/A'):.4f}")
        lines.append(f"    Spearman ρ (held-out): {regression_info.get('spearman_rho', 'N/A'):.4f} "
                     f"(p={regression_info.get('spearman_p', 'N/A'):.4f})")
        lines.append(f"    → {regression_info.get('interpretation', '')}")

    lines.append("\n[7] DPS RANKINGS BY DOMAIN")
    lines.append("-" * 70)
    dps_df = pd.DataFrame([asdict(e) for e in dps_entries])
    for dom in sorted(dps_df["domain"].unique()):
        lines.append(f"\n    Domain: {dom}")
        sub = dps_df[dps_df["domain"] == dom].sort_values("dps_rank")
        lines.append(f"    {'Rank':4s} {'Model':15s} {'Tier':5s} "
                     f"{'Baseline':8s} {'ShiftSens':9s} {'QuantGap':8s} {'DPS':7s}")
        for _, row in sub.iterrows():
            lines.append(
                f"    {int(row['dps_rank']):4d} {row['model_key']:15s} {row['tier']:5s} "
                f"{row['baseline_perf']:8.3f} {row['shift_sensitivity']:9.4f} "
                f"{row['quantization_gap']:8.4f} {row['dps_raw']:+7.4f}"
            )

    lines.append("\n[8] LATENCY-ACCURACY PARETO ANALYSIS")
    lines.append("-" * 70)
    lines.append("    Pareto-optimal: no other model-tier has both higher accuracy AND lower latency")
    lines.append("")
    lines.append(f"    {'Model':15s} {'Tier':5s} {'Avg Acc':8s} {'Avg Lat(ms)':11s} {'Pareto?'}")
    lines.append(f"    {'-'*15} {'-'*5} {'-'*8} {'-'*11} {'-'*7}")
    for _, row in pareto_df.iterrows():
        p_str = "✓ YES" if row["pareto_optimal"] else "no"
        lines.append(
            f"    {row['model_name']:15s} {row['tier']:5s} "
            f"{row['mean_accuracy']:8.3f} {row['mean_latency']:11.1f} {p_str}"
        )

    lines.append("\n" + "=" * 70)
    lines.append("END OF REPORT")
    lines.append("=" * 70)

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# CLI + main
# ---------------------------------------------------------------------------
def parse_args():
    p = argparse.ArgumentParser(description="Phase 3/4 analysis")
    p.add_argument("--baseline",    required=True, help="Path to baseline_results.csv")
    p.add_argument("--shifts",      required=True, help="Path to shift_results.csv")
    p.add_argument("--output-dir",  default="/kaggle/working")
    return p.parse_args()


def main():
    args = parse_args()
    np, pd, stats, LinearRegression, StandardScaler = import_deps()

    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Loading data ...")
    baseline, shifts = load_data(args.baseline, args.shifts, pd)

    print("Running statistical tests ...")
    stats_results = run_statistical_tests(baseline, shifts, stats, np, pd)

    print("Computing degradation profiles ...")
    profiles = compute_degradation_profiles(shifts, np, pd)

    print("Computing DPS ...")
    profiles_df = pd.DataFrame([asdict(p) for p in profiles])
    dps_entries, regression_info = compute_dps(
        baseline, shifts, profiles_df,
        LinearRegression, StandardScaler, np, pd, stats
    )

    print("Computing Pareto analysis ...")
    pareto_df = compute_pareto(baseline, np, pd)

    print("Writing outputs ...")

    # JSON stats
    with open(out_dir / "phase3_stats.json", "w") as f:
        json.dump(stats_results, f, indent=2)

    # Profiles CSV
    profiles_df.to_csv(out_dir / "phase3_profiles.csv", index=False)

    # DPS CSV
    dps_df = pd.DataFrame([asdict(e) for e in dps_entries])
    dps_df.to_csv(out_dir / "phase4_dps.csv", index=False)

    # Regression info JSON
    with open(out_dir / "phase4_regression.json", "w") as f:
        json.dump(regression_info, f, indent=2)

    # Pareto CSV
    pareto_df.to_csv(out_dir / "phase3_pareto.csv", index=False)

    # Human-readable report
    report = write_report(
        stats_results, profiles, dps_entries,
        regression_info, pareto_df, np, pd
    )
    report_path = out_dir / "phase3_report.txt"
    with open(report_path, "w") as f:
        f.write(report)
    print(report)

    print(f"\nOutputs written to {out_dir}:")
    print(f"  phase3_stats.json")
    print(f"  phase3_profiles.csv")
    print(f"  phase4_dps.csv")
    print(f"  phase4_regression.json")
    print(f"  phase3_pareto.csv")
    print(f"  phase3_report.txt")


if __name__ == "__main__":
    main()

Writing phase3_analysis_v2.py


In [4]:
!python phase3_analysis_v2.py \
    --baseline /kaggle/working/baseline_results.csv \
    --shifts   /kaggle/input/datasets/just7nik/r2-d3/shift_results_final.csv \
    --output-dir /kaggle/working

Loading data ...
Baseline: 45 rows | Shifts: 294 rows
Running statistical tests ...
Computing degradation profiles ...
Computing DPS ...
Computing Pareto analysis ...
Writing outputs ...
FRAGILITY UNDER CONSTRAINT — PHASE 3/4 ANALYSIS REPORT

[1] WILCOXON SIGNED-RANK TEST RESULTS
    H0: mean Δacc = 0 (no degradation)
    H1: mean Δacc > 0 (significant degradation)
----------------------------------------------------------------------
    Significant (p<0.05): 12/15 tests

    Model           Shift                        Mean Δacc 95% CI             p-value  Sig?
    --------------- ---------------------------- --------- ------------------ -------- ----
    qwen25          noise                          +0.3511 [+0.2926, +0.4126] 0.0000   YES
    gemma3          prompt_degradation             +0.3161 [+0.1354, +0.5223] 0.0001   YES
    llama31         noise                          +0.2922 [+0.1844, +0.4022] 0.0001   YES
    qwen25          prompt_degradation             +0.2739 [+0.06

In [5]:
with open("/kaggle/working/phase3_report.txt") as f:
    print(f.read())

FRAGILITY UNDER CONSTRAINT — PHASE 3/4 ANALYSIS REPORT

[1] WILCOXON SIGNED-RANK TEST RESULTS
    H0: mean Δacc = 0 (no degradation)
    H1: mean Δacc > 0 (significant degradation)
----------------------------------------------------------------------
    Significant (p<0.05): 12/15 tests

    Model           Shift                        Mean Δacc 95% CI             p-value  Sig?
    --------------- ---------------------------- --------- ------------------ -------- ----
    qwen25          noise                          +0.3511 [+0.2926, +0.4126] 0.0000   YES
    gemma3          prompt_degradation             +0.3161 [+0.1354, +0.5223] 0.0001   YES
    llama31         noise                          +0.2922 [+0.1844, +0.4022] 0.0001   YES
    qwen25          prompt_degradation             +0.2739 [+0.0616, +0.4879] 0.0105   YES
    qwen25          vocab_mismatch                 +0.2483 [+0.1939, +0.3033] 0.0001   YES
    smollm          noise                          +0.2089 [+0.1433, +